In [ ]:
from pyspark.sql import SparkSession

# Start Spark Session
spark = SparkSession.builder.appName("OnlineBankingAnalysis").getOrCreate()

# Load all datasets
loan_df = spark.read.csv("/content/loan.csv", header=True, inferSchema=True)
credit_df = spark.read.csv("/content/credit card.csv", header=True, inferSchema=True)
txn_df = spark.read.csv("/content/txn.csv", header=True, inferSchema=True)

In [ ]:
# Clean loan_df
loan_df = loan_df.withColumnRenamed("Loan Category", "loan_category") \
                 .withColumnRenamed("Loan Amount", "loan_amount") \
                 .withColumnRenamed("Income", "income") \
                 .withColumnRenamed("Returned Cheques", "returned_cheques") \
                 .withColumnRenamed("Marital Status", "marital_status") \
                 .withColumnRenamed("Monthly Expenditure", "monthly_expenditure") \
                 .withColumnRenamed("Eligible For Credit Card", "eligible_credit_card")

# Clean credit_df
credit_df = credit_df.withColumnRenamed("Country", "country") \
                     .withColumnRenamed("Active", "active") \
                     .withColumnRenamed("Eligible", "eligible")

# Clean txn_df
txn_df = txn_df.withColumnRenamed("Account No", "account_no") \
               .withColumnRenamed("Transaction Type", "transaction_type") \
               .withColumnRenamed("Transaction Amount", "transaction_amount") \
               .withColumnRenamed("Balance", "balance") \
               .withColumnRenamed("Transaction Date", "transaction_date")


In [ ]:
from pyspark.sql.functions import col

print("🔹 Loan Data Nulls:")
loan_df.select([col(c).isNull().alias(c) for c in loan_df.columns]).show(5)

print("🔹 Credit Card Data Nulls:")
credit_df.select([col(c).isNull().alias(c) for c in credit_df.columns]).show(5)

print("🔹 Transaction Data Nulls:")
txn_df.select([col(c).isNull().alias(c) for c in txn_df.columns]).show(5)



🔹 Loan Data Nulls:
+-----------+-----+------+----------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|Customer_ID|  Age|Gender|Occupation|marital_status|Family Size|income|Expenditure|Use Frequency|loan_category|loan_amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|
+-----------+-----+------+----------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|      false|false| false|     false|         false|      false| false|      false|        false|        false|      false|  false|       false|           false|             false|
|      false|false| false|     false|         false|      false| false|      false|        false|        false|      false|  false|       false|           false|             false|
|      false|false| false|     false|         false|      false| false|     

In [ ]:
print("🔹 Number of loans in each category:")
loan_df.groupBy("loan_category").count().show()


🔹 Number of loans in each category:
+------------------+-----+
|     loan_category|count|
+------------------+-----+
|           HOUSING|   67|
|        TRAVELLING|   53|
|       BOOK STORES|    7|
|       AGRICULTURE|   12|
|         GOLD LOAN|   77|
|  EDUCATIONAL LOAN|   20|
|        AUTOMOBILE|   60|
|          BUSINESS|   24|
|COMPUTER SOFTWARES|   35|
|           DINNING|   14|
|          SHOPPING|   35|
|       RESTAURANTS|   41|
|       ELECTRONICS|   14|
|          BUILDING|    7|
|        RESTAURANT|   20|
|   HOME APPLIANCES|   14|
+------------------+-----+



In [ ]:
print("🔹 People with loan amount > 1,00,000:")
loan_df.filter(col("loan_amount") > 100000).count()


🔹 People with loan amount > 1,00,000:


0

In [ ]:
print("🔹 People with income > 60,000:")
loan_df.filter(col("income") > 60000).count()


🔹 People with income > 60,000:


198

In [ ]:
print("🔹 People with ≥2 returned cheques and income < 50,000:")
loan_df.filter((col(" Returned Cheque") >= 2) & (col("income") < 50000)).count()

🔹 People with ≥2 returned cheques and income < 50,000:


137

In [ ]:
print("🔹 People with ≥2 returned cheques and are single:")
loan_df.filter((col(" Returned Cheque") >= 2) & (col("marital_status") == "single")).count()

🔹 People with ≥2 returned cheques and are single:


0

In [ ]:
print("🔹 People with expenditure > 50,000 per month:")
loan_df.filter(col("Expenditure") > 50000).count()

🔹 People with expenditure > 50,000 per month:


6

In [ ]:
from pyspark.sql.functions import col, when

# Add eligibility column based on rule: income ≥ 50000, age ≥ 21, returned cheques ≤ 1
loan_df = loan_df.withColumn(
    "eligible_credit_card",
    when(
        (col("income") >= 50000) &
        (col("Age") >= 21) &
        (col(" Returned Cheque") <= 1),
        "yes"
    ).otherwise("no")
)

# Count number of members eligible for credit card
print("🔹 Number of members eligible for credit card:")
loan_df.filter(col("eligible_credit_card") == "yes").count()

🔹 Number of members eligible for credit card:


60

In [ ]:
from pyspark.sql.functions import col

print("📌 Credit card users in Spain:")
credit_df.filter(col("Geography") == "Spain").show()


📌 Credit card users in Spain:
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+--------------+---------------+------+
|RowNumber|CustomerId|  Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+--------------+---------------+------+
|        2|  15647311|     Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|             1|      112542.58|     0|
|        5|  15737888| Mitchell|        850|    Spain|Female| 43|     2|125510.82|            1|             1|        79084.1|     0|
|        6|  15574012|      Chu|        645|    Spain|  Male| 44|     8|113755.78|            2|             0|      149756.71|     1|
|       12|  15737173|  Andrews|        497|    Spain|  Male| 24|     3|      0.0|            2|             0|       76390.01|     0|
|       15|  15600882|   

In [ ]:
from pyspark.sql.functions import col

# Count members who are eligible (CreditScore ≥ 650) and active (IsActiveMember = 1)
eligible_active_count = credit_df.filter(
    (col("CreditScore") >= 650) & (col("IsActiveMember") == 1)
).count()

print(f"📌 Number of members who are eligible and active in the bank: {eligible_active_count}")


📌 Number of members who are eligible and active in the bank: 2672


In [42]:
from pyspark.sql.functions import lower, col, max

txn_df.filter(lower(col("TRANSACTION DETAILS")).rlike(".*(withdrawal|atm|pos|sett).*")) \
      .select(max("WITHDRAWAL AMT").alias("Max_Withdrawal_Amount")) \
      .show()


+---------------------+
|Max_Withdrawal_Amount|
+---------------------+
|                8.0E7|
+---------------------+



In [43]:
from pyspark.sql.functions import min

txn_df.filter(lower(col("TRANSACTION DETAILS")).rlike(".*(withdrawal|atm|pos|sett).*")) \
      .select(min("WITHDRAWAL AMT").alias("Min_Withdrawal_Amount")) \
      .show()


+---------------------+
|Min_Withdrawal_Amount|
+---------------------+
|                 0.01|
+---------------------+



In [44]:
txn_df.filter(lower(col("TRANSACTION DETAILS")).rlike(".*(deposit|credit|income).*")) \
      .select(max("DEPOSIT AMT").alias("Max_Deposit_Amount")) \
      .show()


+------------------+
|Max_Deposit_Amount|
+------------------+
|      1.00150685E8|
+------------------+



In [45]:
txn_df.filter(lower(col("TRANSACTION DETAILS")).rlike(".*(deposit|credit|income).*")) \
      .select(min("DEPOSIT AMT").alias("Min_Deposit_Amount")) \
      .show()


+------------------+
|Min_Deposit_Amount|
+------------------+
|               9.0|
+------------------+



In [47]:
txn_df.groupBy("account_no") \
      .agg({"BALANCE AMT": "sum"}) \
      .withColumnRenamed("sum(BALANCE AMT)", "Total_Balance") \
      .show()

+-------------+--------------------+
|   account_no|       Total_Balance|
+-------------+--------------------+
|409000438611'|-2.49486577068339...|
|     1196711'|-1.60476498101275E13|
|     1196428'| -8.1418498130721E13|
|409000493210'|-3.27584952132095...|
|409000611074'|       1.615533622E9|
|409000425051'|-3.77211841164998...|
|409000405747'|-2.43108047067000...|
|409000362497'| -5.2860004792808E13|
|409000493201'|1.0420831829499985E9|
|409000438620'|-7.12291867951358...|
+-------------+--------------------+



In [49]:
txn_df.groupBy("VALUE DATE") \
      .count() \
      .withColumnRenamed("count", "Transaction_Count") \
      .orderBy("VALUE DATE") \
      .show()

+----------+-----------------+
|VALUE DATE|Transaction_Count|
+----------+-----------------+
|  1-Apr-17|                1|
|  1-Aug-15|               75|
|  1-Aug-16|               85|
|  1-Aug-17|               65|
|  1-Aug-18|              144|
|  1-Dec-15|               96|
|  1-Dec-16|              106|
|  1-Dec-17|               45|
|  1-Dec-18|               97|
|  1-Feb-16|               97|
|  1-Feb-17|               81|
|  1-Feb-18|               87|
|  1-Feb-19|               79|
|  1-Jan-15|                3|
|  1-Jan-16|               59|
|  1-Jan-18|               53|
|  1-Jan-19|               57|
|  1-Jul-15|               25|
|  1-Jul-16|              111|
|  1-Jul-17|              243|
+----------+-----------------+
only showing top 20 rows



In [51]:
txn_df.filter((col("WITHDRAWAL AMT") > 100000)) \
      .select("account_no", "WITHDRAWAL AMT") \
      .distinct() \
      .orderBy("WITHDRAWAL AMT", ascending=False) \
      .show()

+-------------+--------------+
|   account_no|WITHDRAWAL AMT|
+-------------+--------------+
|     1196711'| 4.594475464E8|
|     1196711'| 4.482072231E8|
|409000438620'|         4.0E8|
|409000425051'|        3.54E8|
|     1196711'| 2.671403184E8|
|409000438611'|         2.4E8|
|     1196711'|       2.021E8|
|     1196711'|         2.0E8|
|409000438620'|         2.0E8|
|409000405747'|         1.7E8|
|     1196711'|        1.54E8|
|     1196711'|         1.5E8|
|     1196428'|         1.5E8|
|409000362497'| 1.413662392E8|
|409000362497'| 1.317762365E8|
|409000362497'| 1.316962119E8|
|409000362497'| 1.307105185E8|
|409000438611'|         1.3E8|
|409000362497'| 1.255964844E8|
|409000362497'| 1.221570889E8|
+-------------+--------------+
only showing top 20 rows

